# Figuring out Selenium and bypassing TOS / adult content warnings

In [237]:
# import relevant libraries
import selenium
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import ActionChains
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import re


## trying a thing - selenium (to bypass max retries error)

In [ ]:
from selenium.webdriver.firefox.options import Options

options = Options()
options.add_argument('--disable-blink-features=AutomationControlled')

In [ ]:
# one note - looks like you can do actions.move_to_element(ELEMENT_NAME).perform()

In [ ]:
# testing
driver = webdriver.Firefox()
driver.get('https://archiveofourown.org/works/73925776')

# detect TOS agreement
agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
print("TOS agreement section is present: ", str(agreement_class_present))

# if agreement class is present, find TOS agreement checkboxes / submit button
if agreement_class_present:
    tos_agree = driver.find_element(By.ID, value="tos_agree")
    data_processing_agree = driver.find_element(By.ID, value="data_processing_agree")
    accept_tos_button = driver.find_element(By.ID, value="accept_tos")

    # click checkboxes + accept TOS
    actions = ActionChains(driver)
    actions.move_to_element(tos_agree)
    actions.click(tos_agree)
    actions.move_to_element(data_processing_agree)
    actions.click(data_processing_agree)
    actions.move_to_element(accept_tos_button)
    actions.click(accept_tos_button)
    actions.perform()
else:
    pass

# THIS WORKED

### attempting solutions for MoveTargetOutOfBoundsError
code from: https://testrigor.com/blog/movetargetoutofboundsexception/


In [238]:
# solution attempt: correctly handle dynamic content

"""To handle the dynamic content and the dynamic locations of the elements, use explicit waits to ensure that the page has finished loading and that dynamic elements have stabilized.
"""

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

driver = webdriver.Firefox()
driver.get("https://archiveofourown.org/works/73925776")

# detect TOS agreement
agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
print("TOS agreement section is present: ", str(agreement_class_present))

# if agreement class is present, find TOS agreement checkboxes / submit button
if agreement_class_present:

    # alternative strategy
    wait = WebDriverWait(driver, 10)
    tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

    # click checkboxes + accept TOS
    actions = ActionChains(driver)
    actions.move_to_element(tos_agree).perform()
    actions.click(tos_agree).perform()

    data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
    actions.move_to_element(data_processing_agree).perform()
    actions.click(data_processing_agree).perform()

    accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
    actions.move_to_element(accept_tos_button).perform()
    actions.click(accept_tos_button).perform()

else:
    pass

# detect adult content warning
accept_adult_content_present = len(driver.find_elements(By.LINK_TEXT, 'Yes, Continue')) > 0
print("Adult content warning is present: ", str(accept_adult_content_present))

## THIS is where we're hitting a snag
if accept_adult_content_present:
    wait = WebDriverWait(driver, 15)
    accept_adult_content_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Yes, Continue')))

    # click checkboxes + accept TOS
    actions = ActionChains(driver)
    actions.move_to_element(accept_adult_content_button).perform()
    actions.click(accept_adult_content_button).perform()

else:
    pass

# return page source / html doc
driver.implicitly_wait(10) # seconds
html = driver.page_source

TOS agreement section is present:  True
Adult content warning is present:  True


In [239]:
print(html)

<html lang="en"><head>
    <meta charset="utf-8">
    <meta http-equiv="x-ua-compatible" content="ie=edge">
    <meta name="keywords" content="fanfiction, transformative works, otw, fair use, archive">
    <meta name="language" content="en-US">
    <meta name="subject" content="fandom">
    <meta name="description" content="An Archive of Our Own, a project of the Organization for Transformative Works">
    <meta name="distribution" content="GLOBAL">
    <meta name="classification" content="transformative works">
    <meta name="author" content="Organization for Transformative Works">
  	<meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta name="chrome" content="nointentdetection">
    <meta name="format-detection" content="telephone=no">
    <title>There are times life will rattle your bones - station_oracle - Candela Obscura (Critical Role Web Series) [Archive of Our Own]</title>

    <link rel="stylesheet" type="text/css" media="screen" href="/stylesheets/

In [ ]:
print(type(html))

# Figuring out how to get properly formatted html

In [ ]:
inner_group = driver.find_element(By.ID, value='inner')
print(inner_group)

In [ ]:
# Example code, from: https://pythonexamples.org/python-selenium-get-child-elements/
# Get the div element you are interested in
work_meta_group = driver.find_element(By.CLASS_NAME, 'wrapper')

In [ ]:
work_meta_group_outer = work_meta_group.get_attribute('outerHTML')

In [ ]:
print(work_meta_group_outer)

In [ ]:
print(type(work_meta_group_outer))

In [ ]:
soup = get_soup(work_meta_group_outer)

In [ ]:
print(type(soup))

In [ ]:
print(soup)

In [ ]:
all_dt = get_all_dt(soup)

In [ ]:
print(all_dt)

In [ ]:
all_dt_updated = update_all_dt(all_dt)

In [ ]:
print(all_dt_updated)

In [ ]:
all_dd = get_all_dd(soup)

In [ ]:
all_dd_stripped = strip_dd(all_dd)

In [ ]:
print(all_dd_stripped)

In [ ]:
# dt_group = driver.find_element(By.CLASS_NAME, 'dt')

In [ ]:
# find elements by XPATH
driver.find_elements(By.XPATH, '*')

In [ ]:
# out of curiosity, could we use Selenium to just pull the "work meta group"?

# detect work_meta_group
work_meta_group_present = len(driver.find_elements(By.CLASS_NAME, 'wrapper')) > 0
print("Work meta group is present: ", str(work_meta_group_present))

In [ ]:
# get work meta group
work_meta_group = driver.find_element(By.CLASS_NAME, 'wrapper')

In [ ]:
# get work meta group text
work_meta_group_text = driver.find_element(By.CLASS_NAME, 'wrapper').text

In [ ]:
# split work_meta_group_text by /n
work_meta_group_list = work_meta_group_text.split('\n')

In [ ]:
print(len(work_meta_group_list))

In [ ]:
print(work_meta_group_list)

In [ ]:
# drop unnecessary list items
items_to_drop = ['Archive of Our Own', 'Log In', 'Fandoms', 'Browse', 'Search', 'About', 'Comments Share Download']

# work_meta_group_list1 = work_meta_group_list.drop(items_to_drop)

for item in items_to_drop:
    work_meta_group_list1 = work_meta_group_list.remove(item)

In [ ]:
html1 = driver.find_element(By.XPATH, "/html/body").text

In [ ]:
html = driver.page_source

In [ ]:
print(html)

In [ ]:
print(type(html))

In [ ]:
soup = get_soup(html)

In [ ]:
print(soup)

In [ ]:
print(type(soup))

In [ ]:
all_dt = get_all_dt(soup)

In [ ]:
relationships = soup.find_all("div", {"class": "relationships"})

In [ ]:
print(relationships)

In [ ]:
css_soup = BeautifulSoup('<li class="warnings">', html.parser)
css_soup.find_all('li', class_='warnings')

In [ ]:
css_soup = BeautifulSoup('<p class="body strikeout"></p>', 'html.parser')
css_soup.find_all("p", class_="strikeout")
# [<p class="body strikeout"></p>]

css_soup.find_all("p", class_="body")
# [<p class="body strikeout"></p>]

In [ ]:
all_dt = get_all_dt(soup)

In [ ]:
all_dt_updated = update_all_dt(all_dt)

In [ ]:
print(all_dt_updated)

In [ ]:
all_dd = get_all_dd(soup)

In [ ]:
all_dd_stripped = strip_dd(all_dd)

In [ ]:
all_dd_stripped1 = cleanup_fandom_tags(all_dd_stripped)

In [ ]:
all_dd_stripped2 = cleanup_relationship_tags(all_dd_stripped1)

In [ ]:
all_dd_stripped3 = cleanup_character_tags(all_dd_stripped2)

In [ ]:
all_dd_stripped4 = cleanup_additional_tags(all_dd_stripped3)

In [ ]:
print(all_dd_stripped4)

# Figuring out selenium not providing text

In [ ]:
from selenium import webdriver
from webdriver_manager.firefox import GeckoDriverManager
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
ChromeDriverManager().install()
# driver = webdriver.Chrome(ChromeDriverManager().install())
# driver = webdriver.Firefox(executable_path = GeckoDriverManager().install())

In [ ]:
# WebDriver Chrome
driver = webdriver.Chrome()

# Target URL
driver.get("https://www.geeksforgeeks.org/dsa/competitive-programming-a-complete-guide/")
# To load entire webpage
# time.sleep(5)

# Printing the whole body text
print(driver.find_element(By.XPATH, "/html/body").text)

# Closing the driver
driver.close()

In [ ]:
# importing the modules
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager

# using webdriver for chrome browser
driver = webdriver.Chrome(ChromeDriverManager().install())

# using target url
driver.get(
    "https://www.geeksforgeeks.org/dsa/competitive-programming-a-complete-guide/")

# printing the content of entire page
print(driver.find_element_by_xpath("/html/body").text)

# closing the driver
driver.close()

## drafting solution as function

In [ ]:
def get_fanwork_html(url):
    driver = webdriver.Firefox()
    driver.get(url)

    # detect TOS agreement
    agreement_class_present = len(driver.find_elements(By.CLASS_NAME, value='agreement')) > 0
    print("TOS agreement section is present: ", str(agreement_class_present))

    # if agreement class is present, find TOS agreement checkboxes / submit button
    if agreement_class_present:
        wait = WebDriverWait(driver, 10)
        tos_agree = wait.until(EC.element_to_be_clickable((By.ID, 'tos_agree')))

        # click checkboxes + accept TOS
        actions = ActionChains(driver)
        actions.move_to_element(tos_agree).perform()
        actions.click(tos_agree).perform()

        data_processing_agree = wait.until(EC.element_to_be_clickable((By.ID, 'data_processing_agree')))
        actions.move_to_element(data_processing_agree).perform()
        actions.click(data_processing_agree).perform()

        accept_tos_button = wait.until(EC.element_to_be_clickable((By.ID, 'accept_tos')))
        actions.move_to_element(accept_tos_button).perform()
        actions.click(accept_tos_button).perform()

    else:
        pass

    # detect adult content warning
    accept_adult_content_present = len(driver.find_elements(By.LINK_TEXT, 'Yes, Continue')) > 0
    print("Adult content warning is present: ", str(accept_adult_content_present))

    if accept_adult_content_present:
        wait = WebDriverWait(driver, 15)
        accept_adult_content_button = wait.until(EC.element_to_be_clickable((By.LINK_TEXT, 'Yes, Continue')))

        # click link + accept adult content warning
        actions = ActionChains(driver)
        actions.move_to_element(accept_adult_content_button).perform()
        actions.click(accept_adult_content_button).perform()

    else:
        pass

    # return page source / html doc
    html = driver.page_source
    return html

In [ ]:
# testing function
test_html = get_fanwork_html('https://archiveofourown.org/works/51010177/chapters/128876734')

In [ ]:
print(html)

# trying a thing - requests (to bypass max tries error)

In [ ]:
# code from: https://stackoverflow.com/questions/23013220/max-retries-exceeded-with-url-in-requests
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

test_url = 'https://archiveofourown.org/works/73925776'

session = requests.Session()
retry = Retry(connect=3, backoff_factor=0.5)
adapter = HTTPAdapter(max_retries=retry)
session.mount('http://', adapter)
session.mount('https://', adapter)

session.get(test_url)

# functions from parse_html_docs

In [ ]:
def get_html_doc(url):
    driver = webdriver.Firefox()
    driver.get(url)
    page_source = driver.page_source
    return page_source

In [ ]:
def get_soup(html):
    soup = BeautifulSoup(html, features='html.parser')
    return soup

In [ ]:
def get_all_dt(soup):
    """Takes in BeautifulSoup object from Ao3 fanwork and returns a list of all 'dt' elements of the html document. 'dt' elements represent the metadata categories present for the fanwork."""
    all_dt = soup.find_all('dt')
    # all_dt = list(all_dt)
    return all_dt

In [ ]:
def update_all_dt(all_dt):
    dt_list = list(all_dt)
    dt_dict = {
        '<dt><label for="user_session_login_small">Username or email:</label></dt>':'user_session_login',
        '<dt><label for="user_session_password_small">Password:</label></dt>':'user_session_password',
        '<dt class="rating tags">':'Rating',
        '<dt class="warning tags">':'Archive Warning',
        '<dt class="category tags">':'Categories',
        '<dt class="fandom tags">':'Fandoms',
        '<dt class="relationship tags">':'Relationships',
        '<dt class="character tags">':'Characters',
        '<dt class="freeform tags">':'Additional Tags',
        '<dt class="language">':'Language',
        '<dt class="stats">':'Stats',
        '<dt class="published">':'Date Published',
        '<dt class="status">':'Date Updated',
        '<dt class="words">':'Word Count',
        '<dt class="chapters">':'Chapters',
        '<dt class="comments">':'Comments',
        '<dt class="kudos">':'Kudos',
        '<dt class="bookmarks">':'Bookmarks',
        '<dt class="hits">':'Hits',
        '<dt class="landmark">Note:</dt>':'note',
        '<dt><label for="comment_name_for_211688701">Guest name</label></dt>':'guest_commenter_name',
        '<dt><label for="comment_email_for_211688701">Guest email</label></dt>':'guest_commenter_email'
    }

    for i in range(len(dt_list)):
        dt_list[i] = str(dt_list[i])

    for key in dt_dict.keys():
        for i in range(len(dt_list)):
            key_string = str(key)
            if key_string in dt_list[i]:
                dt_list[i] = dt_dict[key]
            else:
                continue
    return dt_list

In [ ]:
# example from: https://selenium-python.readthedocs.io/api.html#locate-elements-by

menu = driver.find_element(By.CSS_SELECTOR, ".nav")
hidden_submenu = driver.find_element(By.CSS_SELECTOR, ".nav #submenu1")

actions = ActionChains(driver)
actions.move_to_element(menu)
actions.click(hidden_submenu)
actions.perform()

In [ ]:
def get_all_dd(soup):
    """Takes in BeautifulSoup object from Aoe fanwork and returns a list of all 'dd' elements of the html document. 'dd' elements represent the metadata values present in the fanwork."""
    all_dd = soup.find_all('dd')
    # all_dd = list(all_dd)

    # convert dd_list items to strings
    for i in range(len(all_dd)):
        all_dd[i] = str(all_dd[i])

    return all_dd

In [ ]:
def strip_dd(dd_list):
    text_to_strip = ['\n',
                 '<ul class="commas">',
                 '<li>','</li>',
                 '</ul>',
                 '<a class="tag" href="','</a>',
                 '<dd class="rating tags">',
                 '/tags/Not%20Rated/works">',
                 '/tags/General%20Audiences/works">',
                 '/tags/Teen%20And%20Up%20Audiences/works">',
                 '/tags/Mature/works">',
                 '/tags/Explicit/works">',
                 '<dd class="warning tags">',
                 '/tags/No%20Archive%20Warnings%20Apply/works">',
                 '/tags/Graphic%20Depictions%20Of%20Violence/works">',
                 '/tags/Choose%20Not%20To%20Use%20Archive%20Warnings/works">',
                 '/tags/Major%20Character%20Death/works">',
                 '/tags/Rape*s*Non-Con/works">',
                 '/tags/Underage%20Sex/works">',
                 '<dd class="category tags">',
                 '/tags/Gen/works">',
                 '/tags/F*s*F/works">',
                 '/tags/F*s*M/works">F/M',
                 '/tags/M*s*M/works">M/M',
                 '/tags/Multi/works">Multi',
                 '/tags/Other/works">Other',
                 '</dd>',
                 '<dd class="language" lang="en">',
                 '<!-- end of cache -->',
                 '<dd class="stats">',
                 '<dl class="stats">',
                 '<dt class="published">',
                 '</dt><dd class="published">',
                 '<dt class="status">',
                 '</dt><dd class="status">',
                 '<dt class="words">',
                 '</dt><dd class="words">',
                 '<dt class="chapters">',
                 '</dt><dd class="chapters">',
                 '<dt class="comments">',
                 '</dt><dd class="comments">',
                 '<dt class="kudos">',
                 '</dt><dd class="kudos">',
                 '<dt class="bookmarks">',
                 '</dt><dd class="bookmarks">',
                 'bookmarks">',
                 '<a href="/',
                 '<dt class="hits">',
                 '</dt><dd class="hits">',
                 '</dl>',
                 '<dd class="instructions comment_form">',
                 '<dd><input id="',
                 '" name="comment[name]" type="text"/><script>',
                 '/<![CDATA[var ',
                 ' = new LiveValidation',
                 ', { wait: 500, onlyOnBlur: false }',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your name.","validMessage":""})',
                 ';//]]>',
                 '</script>',
                 '<input id="',
                 '" name="comment[email]"',
                 ' type="text"/>',
                 '<script>',
                 '.add(Validate.Presence, {"failureMessage":"Please enter your email address.","validMessage":""})',
                 '<dd><input autocomplete="on" ',
                 'id="user_session_login_small" '
                 ]
    for item in text_to_strip:
        for i in range(len(dd_list)):
            if item in dd_list[i]:
                dd_list[i] = dd_list[i].replace(item, "")
            else:
                continue
    return dd_list

In [ ]:
def cleanup_fandom_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="fandom tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="fandom tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [ ]:
def cleanup_relationship_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="relationship tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="relationship tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [ ]:
def cleanup_character_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="character tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="character tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list

In [ ]:
def cleanup_additional_tags(dd_list):
    for i in range(len(dd_list)):
        if '<dd class="freeform tags">' in dd_list[i]:
            dd_list[i] = dd_list[i].replace('&amp;','&')
            dd_list[i] = dd_list[i].replace('*a*','&')
            dd_list[i] = dd_list[i].replace('%20','')
            dd_list[i] = dd_list[i].replace('<dd class="freeform tags">','')

            dd_list[i] = dd_list[i].split('/tags')

            pattern = re.compile(r"(?<=>)(?!.*>).*")
            for index in range(len(dd_list[i])):
                match = re.search(pattern, dd_list[i][index])
                if match:
                    dd_list[i][index] = match.group(0)
                else:
                    dd_list[i][index] = dd_list[i][index]

        else:
            continue
    return dd_list